# Snowflake Semi-Structured Data Types — Interview Preparation Notes
*Co-authored with CoCo*

**Target Audience:** 3-Year Experienced Snowflake Data Engineer  
**Covers:** VARIANT, OBJECT, ARRAY  
**Optimized For:** Developer Interviews | Data Engineer Interviews | Project Discussions | Client Interviews | Scenario-Based Questions

---

## Schema on Read vs Schema on Write

- **Schema-on-read** stores raw data first and applies the schema during analysis or querying, providing flexibility for different use cases.
- **Schema-on-write** applies the data schema while loading data, ensuring data quality and consistency before storage.

## VARIANT vs OBJECT

- **VARIANT** is the most flexible data type in Snowflake. It can store objects, arrays, and scalar values.
- **OBJECT** stores JSON key-value structures only.

# 1. VARIANT Data Type

## Concept Overview

### Definition
VARIANT is a universal semi-structured data type in Snowflake that can store any type of data: strings, numbers, booleans, dates, arrays, objects, and NULL. It can hold up to **16 MB** of compressed data per value.

### Why Snowflake Introduced It
- To handle **schema-on-read** scenarios where the data structure is not known in advance
- To natively support JSON, Avro, Parquet, ORC, and XML ingestion without pre-defining schemas
- To enable **schema evolution** without ALTER TABLE statements
- To bridge structured and semi-structured data in a single platform

### When to Use It
- Ingesting raw JSON from APIs, Kafka, S3
- Storing event data with variable attributes
- Landing zone / raw layer in data lakehouse architecture
- Audit logging where payload structures change frequently
- Any scenario where schema is unpredictable or evolving

### Key Characteristics
| Feature | Detail |
|---------|--------|
| Max Size | 16 MB (compressed) per value |
| Storage | Columnar, with automatic metadata extraction |
| Supports | JSON, Avro, Parquet, ORC, XML |
| Null Handling | Distinguishes between SQL NULL and JSON null |
| Type Preservation | Stores original data types internally |
| Indexing | Automatic pruning on extracted columns |

---

## Internal Working

### How Snowflake Stores VARIANT Internally
1. **Columnar Storage:** Even though VARIANT looks like a single column, Snowflake internally decomposes it into a columnar format
2. **Metadata Extraction:** Snowflake automatically extracts statistics (min/max) for the first ~200 top-level keys
3. **Micro-partitions:** Data is stored in compressed micro-partitions with metadata for partition pruning
4. **Type Detection:** Snowflake detects and optimizes storage for common types (numbers stored as numbers, not strings)

### Compression Benefits
- Repetitive keys are compressed efficiently (column-level compression)
- Automatic dictionary encoding for repeated string values
- Typically achieves **3x-10x compression** on raw JSON

### Performance Considerations
- Queries on top-level keys perform nearly as well as native columns
- Deeply nested access (>3 levels) may show performance degradation
- Partition pruning works on frequently accessed top-level keys
- Consider flattening frequently queried nested paths into native columns

> **🔥 Most Asked Interview Question:** "How does Snowflake store VARIANT data internally — is it stored as a string or differently?"
>
> **Ideal Answer:** Snowflake does NOT store VARIANT as a plain string. It decomposes the semi-structured data into an optimized columnar format within micro-partitions. It automatically extracts metadata and statistics for top-level keys, enabling partition pruning. This is why querying top-level VARIANT keys performs comparably to querying native relational columns.

## VARIANT — Syntax & Examples

### Table Creation
```sql
-- Simple table with VARIANT column
CREATE OR REPLACE TABLE raw_events (
    event_id INT AUTOINCREMENT,
    event_timestamp TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    raw_payload VARIANT,
    source_system VARCHAR(50)
);

-- Landing table for JSON ingestion
CREATE OR REPLACE TABLE raw_json_landing (
    filename VARCHAR,
    file_row_number INT,
    raw_data VARIANT,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
```

### Insert Statements
```sql
-- Insert using PARSE_JSON
INSERT INTO raw_events (raw_payload, source_system)
SELECT PARSE_JSON('{
    "user_id": 1001,
    "event": "purchase",
    "amount": 299.99,
    "items": ["laptop_bag", "mouse"],
    "metadata": {
        "browser": "Chrome",
        "os": "Windows",
        "ip": "192.168.1.1"
    }
}'), 'web_app';

-- Insert using OBJECT_CONSTRUCT
INSERT INTO raw_events (raw_payload, source_system)
SELECT OBJECT_CONSTRUCT(
    'user_id', 2001,
    'event', 'login',
    'timestamp', CURRENT_TIMESTAMP(),
    'metadata', OBJECT_CONSTRUCT('device', 'mobile', 'app_version', '3.2.1')
), 'mobile_app';

-- Load from staged JSON file
COPY INTO raw_json_landing (filename, file_row_number, raw_data, loaded_at)
FROM (
    SELECT
        METADATA$FILENAME,
        METADATA$FILE_ROW_NUMBER,
        $1,
        CURRENT_TIMESTAMP()
    FROM @my_stage/json_files/
)
FILE_FORMAT = (TYPE = 'JSON');
```

### Querying Data
```sql
-- Access top-level key (dot notation)
SELECT
    raw_payload:user_id::INT AS user_id,
    raw_payload:event::VARCHAR AS event_type,
    raw_payload:amount::FLOAT AS purchase_amount
FROM raw_events;

-- Access nested key
SELECT
    raw_payload:metadata.browser::VARCHAR AS browser,
    raw_payload:metadata.os::VARCHAR AS operating_system
FROM raw_events;

-- Access array element
SELECT
    raw_payload:items[0]::VARCHAR AS first_item,
    raw_payload:items[1]::VARCHAR AS second_item
FROM raw_events;
```

# 2. OBJECT Data Type

## Concept Overview

### Definition
OBJECT is a semi-structured data type that stores **key-value pairs** (similar to a JSON object or Python dictionary). Keys are always strings; values can be any data type including nested OBJECTs and ARRAYs.

### Why Snowflake Introduced It
- To provide a native type specifically for key-value pair structures
- To enable typed semi-structured data (structured OBJECT with defined schema)
- To support complex data modeling within Snowflake

### When to Use It
- Storing configuration settings
- Representing entity attributes that vary by type
- When you need a structured key-value store within a column
- Metadata storage
- When the schema of the object is known (structured OBJECT)

### Key Characteristics
| Feature | Detail |
|---------|--------|
| Keys | Always VARCHAR (strings) |
| Values | Any valid Snowflake type |
| Ordered | Keys are NOT guaranteed to be ordered |
| Duplicates | Duplicate keys are not allowed |
| Max Size | 16 MB (same as VARIANT) |
| Structured | Can define explicit key-value schema |

### OBJECT vs Structured OBJECT
```sql
-- Unstructured OBJECT (flexible, any keys)
CREATE TABLE t1 (config OBJECT);

-- Structured OBJECT (defined schema, type-safe)
CREATE TABLE t2 (
    address OBJECT(
        street VARCHAR,
        city VARCHAR,
        state VARCHAR(2),
        zip VARCHAR(10)
    )
);
```

---

## Syntax & Examples

### Table Creation
```sql
CREATE OR REPLACE TABLE app_configurations (
    app_name VARCHAR(100),
    config OBJECT,
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Structured OBJECT
CREATE OR REPLACE TABLE customer_addresses (
    customer_id INT,
    address OBJECT(
        street VARCHAR,
        city VARCHAR,
        state VARCHAR(2),
        zip VARCHAR(10),
        country VARCHAR(50)
    )
);
```

### Insert Statements
```sql
-- Using OBJECT_CONSTRUCT
INSERT INTO app_configurations (app_name, config)
SELECT 'payment_service', OBJECT_CONSTRUCT(
    'timeout_ms', 3000,
    'retry_count', 3,
    'base_url', 'https://api.payment.com',
    'features', OBJECT_CONSTRUCT(
        'fraud_detection', TRUE,
        'auto_retry', TRUE
    )
);

-- Using PARSE_JSON (returns VARIANT, implicitly stored as OBJECT)
INSERT INTO app_configurations (app_name, config)
SELECT 'notification_service', PARSE_JSON('{
    "email_enabled": true,
    "sms_enabled": false,
    "max_retries": 5
}');
```

### Querying
```sql
SELECT
    app_name,
    config:timeout_ms::INT AS timeout,
    config:retry_count::INT AS retries,
    config:features.fraud_detection::BOOLEAN AS fraud_enabled
FROM app_configurations;
```

# 3. ARRAY Data Type

## Concept Overview

### Definition
ARRAY is a semi-structured data type that stores an **ordered list of values**. Values can be of any type, including nested ARRAYs and OBJECTs. Arrays are zero-indexed.

### Why Snowflake Introduced It
- To natively represent lists, collections, and repeated elements
- To handle multi-valued attributes without separate junction tables
- To support JSON arrays directly
- To enable typed arrays with explicit element types

### When to Use It
- Storing tags, labels, categories for an entity
- Representing order line items
- Multi-valued attributes (phone numbers, email addresses)
- Time-series data points
- Storing hierarchical paths

### Key Characteristics
| Feature | Detail |
|---------|--------|
| Indexing | Zero-based |
| Ordering | Maintains insertion order |
| Max Size | 16 MB (same as VARIANT) |
| Elements | Can be heterogeneous (mixed types) |
| Nesting | Supports nested arrays and objects |
| Structured | Can define explicit element type |

---

## Syntax & Examples

### Table Creation
```sql
CREATE OR REPLACE TABLE product_catalog (
    product_id INT,
    product_name VARCHAR(200),
    tags ARRAY,
    pricing_history ARRAY,
    attributes VARIANT
);

-- Structured ARRAY (typed elements)
CREATE OR REPLACE TABLE employee_skills (
    employee_id INT,
    skills ARRAY(VARCHAR)
);
```

### Insert Statements
```sql
-- Using ARRAY_CONSTRUCT
INSERT INTO product_catalog (product_id, product_name, tags, pricing_history)
SELECT
    101,
    'Wireless Mouse',
    ARRAY_CONSTRUCT('electronics', 'peripherals', 'wireless', 'office'),
    ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('date', '2024-01-01', 'price', 29.99),
        OBJECT_CONSTRUCT('date', '2024-06-01', 'price', 24.99),
        OBJECT_CONSTRUCT('date', '2024-12-01', 'price', 19.99)
    );

-- Using PARSE_JSON
INSERT INTO product_catalog (product_id, product_name, tags)
SELECT 102, 'USB Cable', PARSE_JSON('["electronics", "cables", "accessories"]');
```

### Querying
```sql
-- Access by index
SELECT
    product_name,
    tags[0]::VARCHAR AS primary_tag,
    tags[1]::VARCHAR AS secondary_tag,
    ARRAY_SIZE(tags) AS total_tags
FROM product_catalog;

-- Access nested objects in array
SELECT
    product_name,
    pricing_history[0]:price::FLOAT AS initial_price,
    pricing_history[ARRAY_SIZE(pricing_history)-1]:price::FLOAT AS current_price
FROM product_catalog;
```

# 4. Querying Semi-Structured Data — Complete Guide

## Dot Notation
Used to access top-level and nested keys directly.
```sql
SELECT
    payload:customer.name::VARCHAR AS customer_name,
    payload:customer.address.city::VARCHAR AS city
FROM orders;
```

## Bracket Notation
Used when keys contain special characters, spaces, or are dynamic.
```sql
-- Keys with special characters
SELECT
    payload['first-name']::VARCHAR AS first_name,
    payload['email@domain']::VARCHAR AS email
FROM users;

-- Numeric keys or array access
SELECT
    payload['items'][0]::VARCHAR AS first_item
FROM orders;
```

## Path Expressions with GET_PATH
```sql
-- Using GET_PATH for deep access
SELECT
    GET_PATH(payload, 'customer.address.zip')::VARCHAR AS zip_code,
    GET_PATH(payload, 'items[0].product_name')::VARCHAR AS first_product
FROM orders;
```

## Accessing Nested Elements
```sql
-- Multi-level nesting
SELECT
    raw:response.data.user.profile.preferences.theme::VARCHAR AS theme,
    raw:response.data.user.profile.preferences.language::VARCHAR AS lang
FROM api_responses;
```

## Accessing Arrays
```sql
-- Direct index access
SELECT tags[0]::VARCHAR AS first_tag FROM products;

-- Array size
SELECT ARRAY_SIZE(tags) AS tag_count FROM products;

-- Check if array contains a value
SELECT * FROM products
WHERE ARRAY_CONTAINS('electronics'::VARIANT, tags);
```

## Type Casting (Critical for Interviews)
```sql
-- VARIANT values must be cast for proper comparisons and operations
SELECT
    payload:amount::NUMBER(10,2) AS amount,        -- Numeric
    payload:name::VARCHAR AS name,                  -- String
    payload:is_active::BOOLEAN AS active,           -- Boolean
    payload:created_at::TIMESTAMP_NTZ AS created,   -- Timestamp
    payload:score::FLOAT AS score                   -- Float
FROM events;
```

> **🔥 Most Asked Interview Question:** "What happens if you don't cast VARIANT values?"
>
> **Ideal Answer:** Without explicit casting, VARIANT values remain as VARIANT type. This causes issues with:
> 1. **Comparisons** — WHERE clause filters may not work as expected
> 2. **Aggregations** — SUM/AVG won't work on VARIANT
> 3. **Joins** — Join keys must be cast to matching types
> 4. **String operations** — Values include quotes (e.g., `"John"` instead of `John`)
>
> Always use `::TYPE` or `CAST()` when extracting VARIANT values.

# 5. Important Functions — Syntax & Examples

## PARSE_JSON
Converts a JSON string into a VARIANT value.
```sql
-- Basic usage
SELECT PARSE_JSON('{"name": "John", "age": 30}') AS json_data;

-- In INSERT
INSERT INTO events (payload)
SELECT PARSE_JSON(column1)
FROM raw_string_table;

-- Handling malformed JSON (use TRY_PARSE_JSON)
SELECT TRY_PARSE_JSON('{invalid json}');  -- Returns NULL instead of error
```

## TO_VARIANT
Converts any scalar value to VARIANT type.
```sql
SELECT
    TO_VARIANT(42) AS num_variant,
    TO_VARIANT('hello') AS str_variant,
    TO_VARIANT(CURRENT_DATE()) AS date_variant,
    TO_VARIANT(TRUE) AS bool_variant;
```

## OBJECT_CONSTRUCT
Creates an OBJECT (key-value pairs) from arguments.
```sql
-- Basic
SELECT OBJECT_CONSTRUCT(
    'name', 'Alice',
    'department', 'Engineering',
    'salary', 95000
) AS employee;

-- Skip NULL values with OBJECT_CONSTRUCT_KEEP_NULL vs OBJECT_CONSTRUCT
SELECT OBJECT_CONSTRUCT('a', 1, 'b', NULL, 'c', 3);  
-- Result: {"a": 1, "c": 3}  (NULLs removed)

SELECT OBJECT_CONSTRUCT_KEEP_NULL('a', 1, 'b', NULL, 'c', 3);  
-- Result: {"a": 1, "b": null, "c": 3}  (NULLs kept)

-- Dynamic construction from columns
SELECT OBJECT_CONSTRUCT(*) AS row_as_json FROM employees LIMIT 5;
```

## ARRAY_CONSTRUCT
Creates an ARRAY from arguments.
```sql
-- Basic
SELECT ARRAY_CONSTRUCT(1, 2, 3, 4, 5) AS numbers;
SELECT ARRAY_CONSTRUCT('a', 'b', 'c') AS letters;

-- Mixed types
SELECT ARRAY_CONSTRUCT(1, 'two', 3.0, TRUE, NULL) AS mixed;

-- Empty array
SELECT ARRAY_CONSTRUCT() AS empty_arr;

-- Compact (removes NULLs)
SELECT ARRAY_CONSTRUCT_COMPACT(1, NULL, 3, NULL, 5) AS no_nulls;
-- Result: [1, 3, 5]
```

## FLATTEN ⭐ (Most Important Function for Interviews)
Converts semi-structured data into a relational representation.
```sql
-- Basic FLATTEN on array
SELECT
    o.order_id,
    f.value:product_id::INT AS product_id,
    f.value:quantity::INT AS quantity,
    f.value:price::FLOAT AS price
FROM orders o,
LATERAL FLATTEN(input => o.order_details:items) f;

-- FLATTEN output columns:
-- SEQ    : Sequence counter
-- KEY    : Key (for objects) or index (for arrays)
-- PATH   : Path to the element
-- INDEX  : Array index (NULL for objects)
-- VALUE  : The value at this position
-- THIS   : The original input to FLATTEN

-- Recursive FLATTEN (flattens all nested levels)
SELECT *
FROM orders,
LATERAL FLATTEN(input => payload, RECURSIVE => TRUE);

-- FLATTEN with OUTER => TRUE (keeps rows even if array is empty/null)
SELECT
    c.customer_id,
    f.value::VARCHAR AS phone_number
FROM customers c,
LATERAL FLATTEN(input => c.phone_numbers, OUTER => TRUE) f;

-- Nested FLATTEN (array of arrays)
SELECT
    d.dept_name,
    team.value:team_name::VARCHAR AS team_name,
    member.value::VARCHAR AS member_name
FROM departments d,
LATERAL FLATTEN(input => d.data:teams) team,
LATERAL FLATTEN(input => team.value:members) member;
```

> **🔥 Most Asked Interview Question:** "Explain FLATTEN with a real example and its output columns."

## GET and GET_PATH
```sql
-- GET: access by key name or index
SELECT
    GET(payload, 'name') AS name,       -- Object key access
    GET(my_array, 0) AS first_elem;     -- Array index access

-- GET_PATH: access deeply nested elements using path string
SELECT
    GET_PATH(payload, 'address.city') AS city,
    GET_PATH(payload, 'orders[0].total') AS first_order_total;
```

## TYPEOF
Returns the type of a VARIANT value as a string.
```sql
SELECT
    TYPEOF(PARSE_JSON('42')),              -- 'INTEGER'
    TYPEOF(PARSE_JSON('"hello"')),         -- 'VARCHAR'
    TYPEOF(PARSE_JSON('true')),            -- 'BOOLEAN'
    TYPEOF(PARSE_JSON('[1,2,3]')),         -- 'ARRAY'
    TYPEOF(PARSE_JSON('{"a":1}')),        -- 'OBJECT'
    TYPEOF(PARSE_JSON('null'));            -- 'NULL_VALUE'
```

## IS_OBJECT, IS_ARRAY, IS_NULL_VALUE
```sql
SELECT
    payload,
    IS_OBJECT(payload) AS is_obj,          -- TRUE if OBJECT
    IS_ARRAY(payload:items) AS is_arr,     -- TRUE if ARRAY
    IS_NULL_VALUE(payload:field) AS is_jnull  -- TRUE if JSON null
FROM events;

-- IMPORTANT: IS_NULL_VALUE vs IS NULL
-- IS_NULL_VALUE() checks for JSON null (the value exists but is null)
-- IS NULL checks for SQL NULL (the key doesn't exist)
SELECT
    CASE
        WHEN payload:field IS NULL THEN 'Key missing'
        WHEN IS_NULL_VALUE(payload:field) THEN 'Key exists but value is JSON null'
        ELSE 'Has value'
    END AS field_status
FROM events;
```

> **🔥 Most Asked Interview Question:** "What is the difference between IS NULL and IS_NULL_VALUE()?"
>
> **Ideal Answer:**
> - `IS NULL` → Returns TRUE when the key does **not exist** in the VARIANT or the entire VARIANT column is SQL NULL
> - `IS_NULL_VALUE()` → Returns TRUE when the key **exists** but its value is explicitly set to JSON `null`
> - Example: `{"a": null}` → `payload:a IS NULL` = FALSE, `IS_NULL_VALUE(payload:a)` = TRUE
> - Example: `{"b": 1}` → `payload:a IS NULL` = TRUE, `IS_NULL_VALUE(payload:a)` = NULL

# 6. Real Project Scenarios

## API Ingestion
```sql
-- Landing table for REST API responses
CREATE TABLE raw_api_responses (
    api_name VARCHAR,
    endpoint VARCHAR,
    response_payload VARIANT,
    http_status INT,
    ingested_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Extract and transform API data
CREATE TABLE dim_customers AS
SELECT
    r.response_payload:id::INT AS customer_id,
    r.response_payload:first_name::VARCHAR AS first_name,
    r.response_payload:last_name::VARCHAR AS last_name,
    r.response_payload:email::VARCHAR AS email,
    r.response_payload:address.city::VARCHAR AS city,
    r.response_payload:address.state::VARCHAR AS state,
    f.value:type::VARCHAR AS phone_type,
    f.value:number::VARCHAR AS phone_number
FROM raw_api_responses r,
LATERAL FLATTEN(input => r.response_payload:phones, OUTER => TRUE) f
WHERE r.api_name = 'customer_service';
```

## Kafka Ingestion
```sql
-- Kafka connector landing table
CREATE TABLE kafka_raw (
    record_metadata VARIANT,  -- Kafka metadata (topic, partition, offset)
    record_content VARIANT,   -- Actual message payload
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Stream + Task for CDC processing
CREATE STREAM kafka_raw_stream ON TABLE kafka_raw;

CREATE TASK process_kafka_events
    WAREHOUSE = etl_wh
    SCHEDULE = '1 MINUTE'
WHEN SYSTEM$STREAM_HAS_DATA('kafka_raw_stream')
AS
MERGE INTO fact_transactions t
USING (
    SELECT
        record_content:transaction_id::VARCHAR AS txn_id,
        record_content:amount::NUMBER(12,2) AS amount,
        record_content:customer_id::INT AS customer_id,
        record_content:timestamp::TIMESTAMP_NTZ AS txn_time
    FROM kafka_raw_stream
    WHERE record_content:event_type::VARCHAR = 'TRANSACTION'
) s ON t.txn_id = s.txn_id
WHEN NOT MATCHED THEN INSERT (txn_id, amount, customer_id, txn_time)
VALUES (s.txn_id, s.amount, s.customer_id, s.txn_time);
```

## Audit Logging
```sql
CREATE TABLE audit_log (
    log_id INT AUTOINCREMENT,
    action VARCHAR(50),
    user_id INT,
    old_values VARIANT,  -- Stores previous state
    new_values VARIANT,  -- Stores new state
    changed_fields ARRAY, -- List of changed column names
    context OBJECT,       -- Session/request context
    logged_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Inserting audit record
INSERT INTO audit_log (action, user_id, old_values, new_values, changed_fields, context)
SELECT
    'UPDATE',
    1001,
    OBJECT_CONSTRUCT('salary', 80000, 'title', 'Engineer'),
    OBJECT_CONSTRUCT('salary', 95000, 'title', 'Senior Engineer'),
    ARRAY_CONSTRUCT('salary', 'title'),
    OBJECT_CONSTRUCT('ip', '10.0.0.1', 'session_id', 'abc123', 'app', 'hr_portal');
```

## Data Lake Architecture (Medallion Pattern)
```sql
-- BRONZE (Raw) - Store everything as VARIANT
CREATE TABLE bronze_events (
    source VARCHAR,
    raw_data VARIANT,
    file_name VARCHAR,
    loaded_at TIMESTAMP_NTZ
);

-- SILVER (Cleaned) - Extract and type-cast
CREATE TABLE silver_events AS
SELECT
    raw_data:event_id::VARCHAR AS event_id,
    raw_data:event_type::VARCHAR AS event_type,
    raw_data:user_id::INT AS user_id,
    raw_data:properties::VARIANT AS properties,  -- Keep flexible part as VARIANT
    raw_data:timestamp::TIMESTAMP_NTZ AS event_timestamp,
    loaded_at
FROM bronze_events
WHERE TRY_CAST(raw_data:event_id AS VARCHAR) IS NOT NULL;

-- GOLD (Business-ready) - Fully structured
CREATE TABLE gold_user_activity AS
SELECT
    user_id,
    event_type,
    COUNT(*) AS event_count,
    MIN(event_timestamp) AS first_seen,
    MAX(event_timestamp) AS last_seen
FROM silver_events
GROUP BY 1, 2;
```

## IoT Data Processing
```sql
CREATE TABLE iot_sensor_readings (
    device_id VARCHAR(50),
    reading VARIANT,
    received_at TIMESTAMP_NTZ
);

-- Sample reading:
-- {"temperature": 72.5, "humidity": 45, "pressure": 1013.25,
--  "location": {"lat": 37.77, "lon": -122.41},
--  "alerts": [{"type": "high_temp", "threshold": 70}]}

-- Query with FLATTEN for alerts
SELECT
    device_id,
    reading:temperature::FLOAT AS temp,
    reading:location.lat::FLOAT AS latitude,
    a.value:type::VARCHAR AS alert_type,
    a.value:threshold::FLOAT AS threshold
FROM iot_sensor_readings,
LATERAL FLATTEN(input => reading:alerts, OUTER => TRUE) a
WHERE reading:temperature::FLOAT > 70;
```

# 7. Common Interview Questions & Answers

## VARIANT — Interview Questions

### Basic Level

**Q: What is VARIANT in Snowflake?**  
A: VARIANT is a universal semi-structured data type that can hold any JSON-compatible data (objects, arrays, strings, numbers, booleans, null). It supports up to 16 MB of compressed data and enables schema-on-read querying.

**Q: How do you insert JSON data into a VARIANT column?**  
A: Use `PARSE_JSON()` to convert a JSON string to VARIANT, or load directly from staged files using `COPY INTO` with a JSON file format. You can also use `OBJECT_CONSTRUCT()` for programmatic construction.

**Q: How do you extract values from VARIANT?**  
A: Use dot notation (`payload:key`), bracket notation (`payload['key']`), or `GET_PATH()`. Always cast the result to the target type using `::TYPE`.

### Intermediate Level

**Q: What is the difference between SQL NULL and JSON null in VARIANT?**  
A: SQL NULL means the key doesn't exist or the column is NULL. JSON null means the key exists but has an explicit null value. Use `IS NULL` for SQL NULL and `IS_NULL_VALUE()` for JSON null.

**Q: How does Snowflake optimize VARIANT queries?**  
A: Snowflake extracts statistics (min/max) for top-level keys in VARIANT columns, enabling micro-partition pruning. It stores data in columnar format internally, not as raw strings. Frequently accessed paths benefit from automatic optimization.

**Q: Can you join on VARIANT columns?**  
A: Yes, but you must cast the extracted values first. Example: `JOIN ON a.payload:id::INT = b.customer_id`. Without casting, the join may produce incorrect results or errors.

### Advanced Level

**Q: What are the performance limitations of VARIANT?**  
A: 
1. Deeply nested queries (>3 levels) are slower than top-level access
2. Only top ~200 keys get automatic statistics extraction
3. Clustering keys cannot directly use VARIANT paths (must create computed columns)
4. Large VARIANT values (approaching 16 MB) degrade INSERT performance
5. Type detection may misinterpret data (e.g., date strings stored as VARCHAR)

**Q: How do you handle schema evolution with VARIANT?**  
A: VARIANT naturally handles schema evolution since it's schema-on-read. New fields appear automatically. For downstream typed tables, use `TRY_CAST` and `IS_NULL_VALUE` to handle missing/new fields gracefully. Implement a silver-layer transformation that accommodates new fields without breaking.

---

## OBJECT — Interview Questions

**Q: What is the difference between VARIANT and OBJECT?**  
A: OBJECT specifically represents key-value pairs (JSON objects). VARIANT is more general — it can be an object, array, scalar, or null. An OBJECT value stored in a VARIANT column is still accessed the same way.

**Q: How do you add a key to an existing OBJECT?**  
A: Use `OBJECT_INSERT(obj, 'new_key', value)` to add a key, or `OBJECT_DELETE(obj, 'key')` to remove one.

**Q: What is a structured OBJECT type?**  
A: Introduced in Snowflake, structured OBJECT defines explicit key names and types at the column level: `OBJECT(name VARCHAR, age INT)`. This provides type safety and better IDE/tool support while maintaining flexibility.

---

## ARRAY — Interview Questions

**Q: How do you check if an array contains a specific value?**  
A: Use `ARRAY_CONTAINS(value::VARIANT, array_column)`. Note the value must be cast to VARIANT.

**Q: How do you remove duplicates from an array?**  
A: Use `ARRAY_DISTINCT(array_column)` to get unique values.

**Q: How do you convert rows back into an array?**  
A: Use `ARRAY_AGG(column)` to aggregate multiple rows into a single array. Pair with `WITHIN GROUP (ORDER BY ...)` for ordered results.

```sql
SELECT
    customer_id,
    ARRAY_AGG(order_id) WITHIN GROUP (ORDER BY order_date) AS all_orders
FROM orders
GROUP BY customer_id;
```

# 8. Scenario-Based Interview Questions & Detailed Answers

## Q: Why choose VARIANT over VARCHAR for storing JSON?

**Answer:**
| Aspect | VARIANT | VARCHAR |
|--------|---------|----------|
| Storage | Optimized columnar decomposition | Stored as raw string |
| Querying | Native dot/bracket notation with pruning | Requires PARSE_JSON() every query |
| Performance | Partition pruning on top-level keys | No pruning, full scan every time |
| Validation | Validates JSON structure on INSERT | No validation, can store invalid JSON |
| Functions | All semi-structured functions work directly | Must parse first |
| Size limit | 16 MB compressed | 16 MB characters |

**Interviewer Expects:** Understanding that VARIANT is not just a convenience — it provides real performance benefits through automatic metadata extraction and partition pruning.

---

## Q: Why store raw data in VARIANT instead of directly loading into structured tables?

**Answer:**
1. **Schema Evolution** — Source schemas change without notice; VARIANT absorbs changes without pipeline failures
2. **Data Preservation** — Raw data acts as a replay source; if transformation logic changes, reprocess from raw
3. **Debugging** — When downstream data looks wrong, raw VARIANT enables root-cause analysis
4. **Multiple Consumers** — Different teams may need different fields; raw allows independent silver layers
5. **Speed to Ingest** — No upfront schema definition needed; start ingesting immediately
6. **Compliance** — Some regulations require storing original unmodified source data

---

## Q: How do you handle schema drift/evolution in production?

**Answer:**
```sql
-- 1. Land raw data in VARIANT (immune to schema changes)
COPY INTO raw_landing FROM @stage FILE_FORMAT = (TYPE = JSON);

-- 2. Use TRY_CAST for safe extraction (handles missing/changed types)
CREATE VIEW silver_customers AS
SELECT
    raw:id::INT AS id,
    COALESCE(raw:name::VARCHAR, raw:full_name::VARCHAR) AS name,  -- Handle renamed field
    TRY_CAST(raw:age AS INT) AS age,  -- NULL if type changed
    raw:new_field::VARCHAR AS new_field,  -- New fields auto-appear
    TYPEOF(raw:status) AS status_type  -- Monitor type changes
FROM raw_landing;

-- 3. Monitor schema changes with a detection query
SELECT DISTINCT
    f.key AS field_name,
    TYPEOF(f.value) AS field_type
FROM raw_landing,
LATERAL FLATTEN(input => raw) f
ORDER BY f.key;
```

---

## Q: How do you flatten nested arrays (array of arrays)?

**Answer:**
```sql
-- Data: {"departments": [{"name": "Eng", "teams": [{"name": "Backend", "members": ["Alice", "Bob"]}]}]}

SELECT
    dept.value:name::VARCHAR AS department,
    team.value:name::VARCHAR AS team,
    member.value::VARCHAR AS member_name
FROM company_data,
LATERAL FLATTEN(input => data:departments) dept,
LATERAL FLATTEN(input => dept.value:teams) team,
LATERAL FLATTEN(input => team.value:members) member;

-- Alternative: RECURSIVE FLATTEN
SELECT
    path, key, value
FROM company_data,
LATERAL FLATTEN(input => data, RECURSIVE => TRUE)
WHERE TYPEOF(value) != 'OBJECT' AND TYPEOF(value) != 'ARRAY';
```

---

## Q: What performance issues can occur with semi-structured data and how to optimize?

**Answer:**

| Issue | Cause | Solution |
|-------|-------|----------|
| Slow queries on nested paths | No pruning beyond top-level | Materialize frequent paths as computed/virtual columns |
| Full table scans | Filtering on deep nested values | Add clustering on extracted columns |
| Large VARIANT values | Too much data per row | Split into multiple columns or tables |
| Type mismatches in joins | VARIANT not cast properly | Always explicit cast with `::TYPE` |
| Memory pressure | FLATTEN on large arrays | Filter before FLATTEN, use LIMIT |
| Stale statistics | Frequent schema changes | Re-cluster periodically |

```sql
-- Optimization: Materialized extracted columns
ALTER TABLE raw_events ADD COLUMN user_id INT
    AS (raw_payload:user_id::INT);  -- Virtual/computed column

-- Optimization: Clustering on extracted value
ALTER TABLE raw_events CLUSTER BY (raw_payload:event_type::VARCHAR);

-- Optimization: Search Optimization Service
ALTER TABLE raw_events ADD SEARCH OPTIMIZATION
    ON EQUALITY(raw_payload:user_id);
```

# 9. Project-Based Interview Questions (STAR Format)

## Q: Describe a project where you used VARIANT.

**Situation:** Our e-commerce platform integrated with 15+ third-party APIs (payment gateways, shipping providers, product feeds) each with different JSON response structures. Schemas changed every 2-3 months without notice.

**Task:** Design a data pipeline that could handle all API responses without breaking when schemas changed, and provide analytics within 15 minutes of data arrival.

**Action:**
1. Created a unified landing table with VARIANT column to store all API responses
2. Implemented Snowpipe for continuous ingestion from S3 (JSON file format)
3. Built silver-layer views using `TRY_CAST` and `COALESCE` for backward-compatible extraction
4. Created a schema detection task that ran daily to identify new/changed fields
5. Used FLATTEN to normalize nested order items into a relational fact table
6. Added Search Optimization on frequently filtered VARIANT paths

**Result:** Zero pipeline failures due to schema changes over 18 months. Reduced data engineering maintenance by 60%. Analytics latency reduced from 4 hours to 12 minutes.

---

## Q: How did you ingest JSON files from S3 into Snowflake?

**Situation:** Daily data dumps of 500+ JSON files (2-5 GB total) from upstream microservices landing in S3.

**Task:** Build an automated, fault-tolerant ingestion pipeline with exactly-once semantics.

**Action:**
```sql
-- 1. External Stage
CREATE STAGE s3_json_stage
    URL = 's3://company-data-lake/json/'
    STORAGE_INTEGRATION = s3_integration
    FILE_FORMAT = (TYPE = JSON, STRIP_OUTER_ARRAY = TRUE);

-- 2. Landing Table
CREATE TABLE raw_json (
    src VARIANT,
    filename VARCHAR DEFAULT METADATA$FILENAME,
    file_row INT DEFAULT METADATA$FILE_ROW_NUMBER,
    load_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 3. Snowpipe for auto-ingestion
CREATE PIPE json_pipe AUTO_INGEST = TRUE AS
COPY INTO raw_json (src, filename, file_row, load_ts)
FROM (
    SELECT $1, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER, CURRENT_TIMESTAMP()
    FROM @s3_json_stage
);

-- 4. Error handling with VALIDATION_MODE
COPY INTO raw_json FROM @s3_json_stage
    VALIDATION_MODE = 'RETURN_ERRORS';
```

**Result:** Fully automated pipeline processing 500+ files daily with 99.9% reliability. Used Snowpipe's exactly-once guarantee to prevent duplicates. Error files routed to a dead-letter queue for investigation.

---

## Q: How did you troubleshoot semi-structured data issues?

**Situation:** Analytics team reported NULL values in dashboard metrics. The source data was a VARIANT column from a Kafka stream.

**Task:** Identify root cause and fix the data quality issue.

**Action:**
1. Used `TYPEOF()` to detect type changes: found that `amount` changed from INTEGER to STRING
2. Used schema detection query to compare current vs historical schemas
3. Fixed extraction with `TRY_CAST` and added data quality checks:

```sql
-- Diagnostic queries
SELECT TYPEOF(payload:amount), COUNT(*)
FROM kafka_raw
GROUP BY 1;
-- Found: INTEGER (95%), VARCHAR (5%) — upstream started quoting numbers

-- Fix: Handle both types
SELECT
    COALESCE(
        TRY_CAST(payload:amount AS NUMBER(12,2)),
        TRY_CAST(payload:amount::VARCHAR AS NUMBER(12,2))
    ) AS amount
FROM kafka_raw;

-- Prevention: Data quality monitor
CREATE TASK schema_monitor SCHEDULE = '60 MINUTE' AS
INSERT INTO schema_audit
SELECT key, TYPEOF(value), COUNT(*), CURRENT_TIMESTAMP()
FROM kafka_raw, LATERAL FLATTEN(input => payload)
WHERE loaded_at > DATEADD('hour', -1, CURRENT_TIMESTAMP())
GROUP BY 1, 2;
```

**Result:** Identified the issue within 30 minutes. Implemented a fix that handled both string and numeric amounts. Added proactive monitoring that caught 3 more schema changes before they caused issues.

# 10. Detailed Comparisons

## VARIANT vs OBJECT

| Aspect | VARIANT | OBJECT |
|--------|---------|--------|
| **What it stores** | Any type (scalar, array, object, null) | Only key-value pairs |
| **Flexibility** | Maximum — holds anything | Key-value only |
| **Type checking** | None (accepts everything) | Validates structure is key-value |
| **Common use** | Raw data landing, general storage | Configuration, attributes, metadata |
| **Structured version** | N/A | `OBJECT(key1 TYPE, key2 TYPE)` |
| **Querying** | Same syntax (`:key`) | Same syntax (`:key`) |
| **Conversion** | `TO_VARIANT(obj)` | `TO_OBJECT(variant)` |
| **Construction** | `PARSE_JSON(...)` | `OBJECT_CONSTRUCT(...)` |

**When to use which:** Use VARIANT when data can be any structure. Use OBJECT (especially structured OBJECT) when you know it's always key-value pairs and want type safety.

---

## VARIANT vs ARRAY

| Aspect | VARIANT | ARRAY |
|--------|---------|-------|
| **What it stores** | Any type | Ordered list of values |
| **Access pattern** | By key or index | By index only |
| **Construction** | `PARSE_JSON` | `ARRAY_CONSTRUCT` |
| **Iteration** | `FLATTEN` | `FLATTEN` |
| **Contains check** | Manual | `ARRAY_CONTAINS()` |
| **Size** | `N/A` | `ARRAY_SIZE()` |
| **Aggregation** | N/A | `ARRAY_AGG()` |

---

## OBJECT vs ARRAY

| Aspect | OBJECT | ARRAY |
|--------|--------|-------|
| **Structure** | Unordered key-value pairs | Ordered indexed elements |
| **Access** | By key name | By numeric index |
| **Keys** | String keys required | No keys (integer indices) |
| **Best for** | Entity attributes, configs | Lists, collections, sequences |
| **Example** | `{"name": "John", "age": 30}` | `["tag1", "tag2", "tag3"]` |
| **Manipulation** | `OBJECT_INSERT`, `OBJECT_DELETE` | `ARRAY_APPEND`, `ARRAY_PREPEND` |

---

## VARIANT vs VARCHAR (Storing JSON)

| Aspect | VARIANT | VARCHAR |
|--------|---------|----------|
| **Storage** | Columnar decomposition, optimized | Raw string bytes |
| **Querying** | Direct path access with pruning | Must `PARSE_JSON()` every time |
| **Validation** | Validates JSON on INSERT | No validation |
| **Performance** | Partition pruning on top keys | No pruning possible |
| **Functions** | All semi-structured functions | String functions only (until parsed) |
| **Cost** | More efficient for repeated queries | Cheaper for write-once-read-never |
| **Type safety** | Preserves types internally | Everything is string |

> **Verdict:** Always use VARIANT for JSON data you intend to query. Only use VARCHAR if you're storing JSON purely for archival with no query needs.

---

## Semi-Structured vs Structured Data

| Aspect | Semi-Structured (VARIANT) | Structured (Native Columns) |
|--------|---------------------------|-----------------------------|
| **Schema** | Schema-on-read | Schema-on-write |
| **Flexibility** | High — adapts to changes | Low — requires ALTER TABLE |
| **Performance** | Good (top-level), degrades for deep nesting | Best — full optimization |
| **Pruning** | Automatic for ~200 top keys | Full pruning on all columns |
| **Clustering** | Cannot cluster directly on VARIANT paths | Full clustering support |
| **Joins** | Requires casting | Native type matching |
| **Best for** | Raw landing, evolving schemas | Analytics, reporting, aggregations |
| **Storage** | Slightly larger | Most compact |

# 11. Best Practices

## When to Use VARIANT
- Raw/landing layer (Bronze) in medallion architecture
- Data from external APIs with unstable schemas
- Event-driven data with variable attributes
- Prototyping — quickly ingest and explore new data sources
- When different records have fundamentally different structures

## When NOT to Use VARIANT
- Final analytics/reporting tables (Gold layer)
- High-performance aggregation queries
- Columns that need to be primary/foreign keys
- When the schema is stable and well-known
- Dimension tables that are frequently joined

## Performance Optimization Tips

1. **Materialize frequently queried paths:**
```sql
ALTER TABLE events ADD COLUMN event_type VARCHAR AS (payload:event_type::VARCHAR);
```

2. **Use Search Optimization on VARIANT paths:**
```sql
ALTER TABLE events ADD SEARCH OPTIMIZATION ON EQUALITY(payload:user_id);
```

3. **Filter BEFORE flattening:**
```sql
-- GOOD: Filter first, then flatten
SELECT f.value:name::VARCHAR
FROM events, LATERAL FLATTEN(input => payload:items) f
WHERE payload:event_type::VARCHAR = 'purchase';  -- Filter BEFORE flatten

-- BAD: Flatten everything, then filter
SELECT f.value:name::VARCHAR
FROM events, LATERAL FLATTEN(input => payload:items) f
WHERE f.value:category::VARCHAR = 'electronics';  -- Flattens ALL rows first
```

4. **Use OUTER => TRUE for LEFT JOIN behavior with FLATTEN**
5. **Avoid SELECT * on tables with large VARIANT columns** — extract only needed paths
6. **Use TRY_PARSE_JSON and TRY_CAST for fault-tolerant pipelines**
7. **Cluster on extracted VARIANT paths for large tables (>1TB)**

## Data Modeling Recommendations

| Layer | Approach |
|-------|----------|
| Bronze/Raw | Single VARIANT column + metadata columns |
| Silver/Cleaned | Extract known fields as typed columns, keep unknown in VARIANT |
| Gold/Business | Fully relational, no VARIANT columns |

## Production Considerations
- Set up data quality checks on extracted VARIANT paths
- Monitor TYPEOF() distributions to detect schema drift
- Implement dead-letter queues for malformed JSON (using TRY_PARSE_JSON)
- Document expected schemas even though VARIANT doesn't enforce them
- Use STRIP_OUTER_ARRAY = TRUE when loading JSON arrays as separate rows

# 12. Common Mistakes (Top 15)

| # | Mistake | Impact | Fix |
|---|---------|--------|-----|
| 1 | Not casting VARIANT values (`payload:id` instead of `payload:id::INT`) | Wrong comparisons, join failures, string quotes in output | Always use `::TYPE` casting |
| 2 | Using `IS NULL` to check for JSON null | Misses rows where key exists with null value | Use `IS_NULL_VALUE()` for JSON null |
| 3 | Storing JSON as VARCHAR instead of VARIANT | No pruning, must parse on every query, no validation | Use VARIANT for any queryable JSON |
| 4 | Not using `OUTER => TRUE` in FLATTEN | Rows with empty/null arrays silently disappear | Use `OUTER => TRUE` for LEFT JOIN behavior |
| 5 | Flattening before filtering | Massive row explosion, slow queries | Filter parent table BEFORE FLATTEN |
| 6 | Not handling type changes from source | NULL values in downstream tables | Use `TRY_CAST` and `COALESCE` |
| 7 | Deeply nesting all data in one VARIANT column | Poor query performance | Extract frequently accessed fields into native columns |
| 8 | Forgetting STRIP_OUTER_ARRAY when loading JSON arrays | Entire array stored as single row | Set `STRIP_OUTER_ARRAY = TRUE` in file format |
| 9 | Case sensitivity — using wrong case for keys | Returns NULL (VARIANT keys are case-sensitive!) | Match exact case: `payload:userId` ≠ `payload:USERID` |
| 10 | Using VARIANT columns as clustering keys | Clustering requires scalar types | Cluster on extracted computed columns |
| 11 | Not using TRY_PARSE_JSON for untrusted input | Pipeline fails on single malformed record | Use `TRY_PARSE_JSON` (returns NULL on failure) |
| 12 | Joining on uncast VARIANT values | Incorrect results or type mismatch errors | Cast both sides to same type |
| 13 | Assuming VARIANT preserves key order | Key order is NOT guaranteed in objects | Never rely on key ordering |
| 14 | Loading large JSON (>16MB) without splitting | INSERT fails with size limit error | Pre-process to split large documents |
| 15 | Not monitoring schema evolution | Silent data quality issues | Run periodic TYPEOF() distribution checks |

> **🔥 Most Asked Interview Question:** "What is the biggest mistake you've seen with semi-structured data in Snowflake?"
>
> **Ideal Answer:** Case sensitivity is the most insidious — VARIANT key access is case-sensitive unlike regular SQL columns. Developers assume `payload:user_id` and `payload:User_ID` are the same, but they return different values (or NULL). I've seen entire dashboards show zeros because of a case mismatch after an upstream system changed their JSON key casing.

# 13. Interview Cheat Sheet — Final Revision

## One-Line Definitions
| Type | Definition |
|------|------------|
| **VARIANT** | Universal type that stores any semi-structured data (scalar, object, array, null) up to 16MB |
| **OBJECT** | Stores key-value pairs where keys are strings and values are any type |
| **ARRAY** | Stores an ordered, zero-indexed list of values of any type |

## Key Functions Quick Reference
| Function | Purpose | Example |
|----------|---------|----------|
| `PARSE_JSON()` | String → VARIANT | `PARSE_JSON('{"a":1}')` |
| `TRY_PARSE_JSON()` | Safe string → VARIANT | Returns NULL on invalid JSON |
| `OBJECT_CONSTRUCT()` | Build OBJECT from pairs | `OBJECT_CONSTRUCT('k', 'v')` |
| `ARRAY_CONSTRUCT()` | Build ARRAY from values | `ARRAY_CONSTRUCT(1, 2, 3)` |
| `FLATTEN()` | Explode array/object to rows | `LATERAL FLATTEN(input => col)` |
| `GET_PATH()` | Access nested path by string | `GET_PATH(v, 'a.b.c')` |
| `TYPEOF()` | Get VARIANT value's type | Returns 'INTEGER', 'VARCHAR', etc. |
| `IS_NULL_VALUE()` | Check for JSON null | Distinguishes from SQL NULL |
| `ARRAY_SIZE()` | Count array elements | `ARRAY_SIZE(my_array)` |
| `ARRAY_CONTAINS()` | Check membership | `ARRAY_CONTAINS('x'::VARIANT, arr)` |
| `OBJECT_KEYS()` | Get all keys from object | Returns ARRAY of key names |
| `ARRAY_AGG()` | Rows → array | `ARRAY_AGG(col) WITHIN GROUP (ORDER BY ...)` |

## Essential Syntax Patterns
```sql
-- Extract and cast
payload:key::VARCHAR

-- Nested access
payload:level1.level2.level3::TYPE

-- Array access
payload:array_field[0]::TYPE

-- FLATTEN pattern
SELECT t.id, f.value:field::TYPE
FROM table t, LATERAL FLATTEN(input => t.variant_col:array_key) f;

-- Safe extraction
COALESCE(TRY_CAST(payload:field AS INT), 0)

-- Schema detection
SELECT DISTINCT f.key, TYPEOF(f.value)
FROM table, LATERAL FLATTEN(input => variant_col) f;
```

## Top 10 Most Frequently Asked Questions
1. What is VARIANT and how does Snowflake store it internally?
2. Difference between IS NULL and IS_NULL_VALUE()?
3. How does FLATTEN work? Explain with an example.
4. How do you handle schema evolution with VARIANT?
5. VARIANT vs VARCHAR for JSON storage — which is better and why?
6. How do you optimize queries on VARIANT columns?
7. What is dot notation vs bracket notation?
8. How to flatten nested arrays (multiple LATERAL FLATTENs)?
9. What are the FLATTEN output columns (SEQ, KEY, PATH, INDEX, VALUE, THIS)?
10. How did you use semi-structured data in your project? (STAR format)

## Quick Comparison Table
| Feature | VARIANT | OBJECT | ARRAY | VARCHAR (JSON) |
|---------|---------|--------|-------|----------------|
| Holds any type | ✅ | ❌ (key-value only) | ❌ (list only) | ✅ (as string) |
| Native querying | ✅ | ✅ | ✅ | ❌ (must parse) |
| Partition pruning | ✅ | ✅ | ✅ | ❌ |
| Type validation | ❌ | Structured: ✅ | Structured: ✅ | ❌ |
| Max size | 16 MB | 16 MB | 16 MB | 16 MB |
| Best for | Raw data | Configs/attributes | Lists/tags | Archival only |

## Project Discussion Points (Have Ready)
1. "In our Bronze layer, we use VARIANT to store raw API responses for schema resilience"
2. "We use FLATTEN with OUTER => TRUE to normalize nested order items"
3. "We implemented schema drift detection using TYPEOF() and LATERAL FLATTEN"
4. "We materialized hot paths as virtual columns and added Search Optimization"
5. "We use TRY_PARSE_JSON and dead-letter tables for fault-tolerant ingestion"
6. "Our Kafka pipeline uses VARIANT landing → Stream → Task → typed silver tables"

---

**Remember for Interviews:**
- Always mention **real numbers** (rows processed, latency reduced, % improvement)
- Show awareness of **performance trade-offs** (VARIANT convenience vs native column speed)
- Demonstrate **production thinking** (error handling, monitoring, schema drift)
- Know the **FLATTEN output columns** by heart (SEQ, KEY, PATH, INDEX, VALUE, THIS)
- Practice explaining the **IS NULL vs IS_NULL_VALUE** difference — it comes up in 80%+ of interviews

---
*End of Notes — Good luck with your interviews!*

In [ ]:
-----------------------------------------------------
----------------------- VARIANT ---------------------
-----------------------------------------------------
create or replace table my_complex_tbl
(
    var variant
);

-- Error : JSON object literal is not allowed inside VALUES.
insert into my_complex_tbl
values ({"name":"Aditya", "age":"27", "address":{"city":"rohini", "state":"delhi", "pin":"123211"}});

-- Error: Data is varchar, can't be stored in variant
insert into my_complex_tbl
values ('{"name":"Aditya", "age":"27", "address":{"city":"rohini", "state":"delhi", "pin":"123211"}}');

-- Error: Parse_json is not allowed in Values
insert into my_complex_tbl
values (parse_json('{"name":"Aditya", "age":"27", "address":{"city":"rohini", "state":"delhi", "pin":"123211"}}'));

-- Correct: Use Parse_json
insert into my_complex_tbl
select parse_json('{"name":"Aditya", "age":"27", "address":{"city":"rohini", "state":"delhi", "pin":"123211"}}');

select * from my_complex_tbl;

select var:name::varchar, var:age::int, var:address, var:address.city, var:address.state::varchar from my_complex_tbl;


-----------------------------------------------------
------------------------ OBJECT ---------------------
-----------------------------------------------------
-- 1. Creating objects
select {*} from (
    select 'Aditya' as name
    UNION
    select 'Tarun' as name
    UNION
    select 'Rahul' as name
)

-- 2. Generating object from table rows
CREATE OR REPLACE TABLE demo_ca_provinces (province VARCHAR, capital VARCHAR);
INSERT INTO demo_ca_provinces (province, capital) VALUES
  ('Ontario', 'Toronto'),
  ('British Columbia', 'Victoria'),
  ('Aditya', 'Don'),
  ('Tarun', 'QA Master');

SELECT province, capital FROM demo_ca_provinces ORDER BY province; -- table data
SELECT {*} FROM demo_ca_provinces;  -- generating objects


-- 3. Storing object data into object column
CREATE OR REPLACE TABLE my_object_table (object_column OBJECT);

INSERT INTO my_object_table (object_column)
  SELECT OBJECT_CONSTRUCT('thirteen', 13::VARIANT, 'zero', 0::VARIANT);

INSERT INTO my_object_table (object_column)
  SELECT { 'PROVINCE': 'Alberta'::VARIANT , 'CAPITAL': 'Edmonton'::VARIANT };

INSERT INTO my_object_table (object_column)
  SELECT OBJECT_CONSTRUCT('PROVINCE', 'Manitoba'::VARIANT , 'CAPITAL', 'Winnipeg'::VARIANT );

INSERT INTO my_object_table (object_column)
  SELECT OBJECT_CONSTRUCT('PROVINCE', 'Manitoba_sys' , 'CAPITAL', 'Winnipeg_Sys');

INSERT INTO my_object_table (object_column)
  SELECT parse_json('{"name":"Aditya", "age":"27", "address":{"city":"rohini", "state":"delhi", "pin":"123211"}}');

SELECT * FROM my_object_table;

SELECT object_column['thirteen'], object_column['PROVINCE'] from my_object_table;